# 03 — SHAP Explanations

Computes **TreeSHAP attributions** for every replica of every window pair, restricted to the flagged instances F_{A,B}.

**Input:** `data/models/pair_{i}/` (from notebook 02)  
**Output per pair:** `data/shap/pair_{i}/shap_A.npy` and `shap_B.npy`  
- Shape: `(R, |F_{A,B}|, p)` — replicas × flagged instances × features

---

**TreeSHAP note (§3.3):** TreeSHAP is deterministic for a fixed model and background dataset. There is therefore no explainer stochasticity for SHAP — this notebook verifies that claim and reports the (expected) zero stochasticity diagnostic.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import shap
from pathlib import Path

WORKSPACE = Path(r'C:\Users\Frewl\OneDrive - KU Leuven\KU Leuven\Thesis\Shoppers_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'
MODEL_DIR = WORKSPACE / 'data' / 'models'
SHAP_DIR  = WORKSPACE / 'data' / 'shap'
SHAP_DIR.mkdir(parents=True, exist_ok=True)

print(f'SHAP version: {shap.__version__}')

In [ ]:
# Load data and window config
X      = pd.read_parquet(PROC_DIR / 'X.parquet').values.astype(np.float32)
with open(PROC_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R     = config['parameters']['R']
pairs = config['pairs']

print(f'X: {X.shape}, features: {len(feature_names)}')
print(f'R={R}, {len(pairs)} pairs')

## SHAP computation loop

For each pair and each replica:
1. Load the model and scaler.
2. Scale the flagged instances using the pair's reference scaler.
3. Compute TreeSHAP values via `shap.TreeExplainer`.

Results are saved immediately after each pair to allow resuming.

In [ ]:
for p in pairs:
    pid       = p['pair_id']
    pair_dir  = MODEL_DIR / f'pair_{pid:02d}'
    shap_dir  = SHAP_DIR  / f'pair_{pid:02d}'

    shap_A_path = shap_dir / 'shap_A.npy'
    shap_B_path = shap_dir / 'shap_B.npy'

    if shap_A_path.exists() and shap_B_path.exists():
        print(f'Pair {pid:02d}: SHAP already computed, skipping.')
        continue

    print(f'\n── Pair {pid:02d} ──────────────────────────────────────────────')
    shap_dir.mkdir(parents=True, exist_ok=True)

    # Load scaler and prediction data
    scaler   = joblib.load(pair_dir / 'scaler.joblib')
    pred_data = np.load(pair_dir / 'predictions.npz')
    flagged_local_idx = pred_data['flagged_idx']   # positions within idx_eval
    idx_eval = np.array(p['idx_eval'], dtype=np.int64)

    # Global row indices of flagged instances
    flagged_global_idx = idx_eval[flagged_local_idx]
    X_flagged = scaler.transform(X[flagged_global_idx])  # (|F|, p)

    n_flagged = X_flagged.shape[0]
    n_feat    = X_flagged.shape[1]
    print(f'  Flagged instances: {n_flagged:,}  Features: {n_feat}')

    if n_flagged == 0:
        print('  WARNING: no flagged instances — skipping.')
        continue

    # ── SHAP for replicas of window A ─────────────────────────────────
    shap_A = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
    for r in range(R):
        model = joblib.load(pair_dir / 'replicas_A' / f'model_r{r}.joblib')
        explainer = shap.TreeExplainer(model)
        vals = explainer.shap_values(X_flagged)
        # For binary classification XGBoost returns a 2D array directly
        # or a list [neg_class, pos_class]; we want positive class
        if isinstance(vals, list):
            vals = vals[1]
        shap_A[r] = vals.astype(np.float32)
        print(f'  A replica {r}: SHAP computed  |mean|={np.abs(shap_A[r]).mean():.5f}')

    # ── SHAP for replicas of window B ─────────────────────────────────
    shap_B = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
    for r in range(R):
        model = joblib.load(pair_dir / 'replicas_B' / f'model_r{r}.joblib')
        explainer = shap.TreeExplainer(model)
        vals = explainer.shap_values(X_flagged)
        if isinstance(vals, list):
            vals = vals[1]
        shap_B[r] = vals.astype(np.float32)
        print(f'  B replica {r}: SHAP computed  |mean|={np.abs(shap_B[r]).mean():.5f}')

    # ── Sanity check: SHAP values should sum to prediction − base_value ──
    # Check one replica of A
    model_A0 = joblib.load(pair_dir / 'replicas_A' / 'model_r0.joblib')
    exp0 = shap.TreeExplainer(model_A0)
    preds_raw = model_A0.predict(X_flagged, output_margin=True)  # log-odds
    shap_sum  = shap_A[0].sum(axis=1) + exp0.expected_value
    max_err = np.abs(shap_sum - preds_raw).max()
    print(f'  Sanity (A r0): max |SHAP_sum - log_odds| = {max_err:.6f}')
    if max_err > 0.01:
        print('  WARNING: SHAP consistency error too large!')
    else:
        print('  Sanity check passed.')

    # ── Save ──────────────────────────────────────────────────────────
    np.save(shap_A_path, shap_A)
    np.save(shap_B_path, shap_B)
    print(f'  Saved shap_A {shap_A.shape}, shap_B {shap_B.shape}')

print('\n✓ All SHAP values computed.')

## SHAP stochasticity diagnostic (§3.9)

TreeSHAP is deterministic — running the explainer twice on the same model and data returns identical values. We verify this for pair 0 as a diagnostic.

In [ ]:
# Use pair 0, best-performing replica of A
pid = 0
pair_dir = MODEL_DIR / f'pair_{pid:02d}'
pred_data = np.load(pair_dir / 'predictions.npz')
scaler = joblib.load(pair_dir / 'scaler.joblib')

idx_eval = np.array(pairs[pid]['idx_eval'], dtype=np.int64)
flagged_local_idx = pred_data['flagged_idx']
flagged_global_idx = idx_eval[flagged_local_idx]
X_flagged = scaler.transform(X[flagged_global_idx])

# Best replica of A = highest PR-AUC
preds_A = pred_data['preds_A']   # (R, n_eval)
from sklearn.metrics import average_precision_score
Y_eval = pred_data['Y_eval']
best_r = max(range(R), key=lambda r: average_precision_score(Y_eval, preds_A[r]))
print(f'Best replica of A: r={best_r}')

model_best = joblib.load(pair_dir / 'replicas_A' / f'model_r{best_r}.joblib')
exp = shap.TreeExplainer(model_best)

# Run explainer twice
run1 = exp.shap_values(X_flagged)
if isinstance(run1, list): run1 = run1[1]
run2 = exp.shap_values(X_flagged)
if isinstance(run2, list): run2 = run2[1]

stoch_cosine = np.abs(run1 - run2).max()
print(f'Max absolute difference between two SHAP runs: {stoch_cosine:.2e}')
print('TreeSHAP is deterministic ✓' if stoch_cosine < 1e-6 else 'WARNING: non-zero stochasticity!')

## Quick SHAP summary (pair 0)

In [ ]:
import matplotlib.pyplot as plt

shap_A_0 = np.load(SHAP_DIR / 'pair_00' / 'shap_A.npy')   # (R, |F|, p)

# Replica-averaged SHAP for pair 0
phi_bar_A = shap_A_0.mean(axis=0)   # (|F|, p)

# Global importance: mean |φ| per feature
global_imp = np.abs(phi_bar_A).mean(axis=0)
top_k = 20
top_idx = np.argsort(global_imp)[::-1][:top_k]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    [feature_names[i] for i in top_idx[::-1]],
    global_imp[top_idx[::-1]],
    color='steelblue'
)
ax.set_title(f'Top {top_k} global SHAP importances (pair 0, model A)')
ax.set_xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.savefig(SHAP_DIR / 'global_importance_pair00_A.png', dpi=120)
plt.show()